# Exterior OPNS: spanning and intersecting points, lines, and circles

The exterior algebras `exterior3` and `exterior4` from `algpy.lattices` have no
metric at all. They are lattices: the exterior product `|` (join) and the
regressive product `&` (meet), connected by the Poincaré duals `ldual()` and
`rdual()`. That is enough to construct geometric objects from points and to
intersect them:

- a point is the right dual of its lifted homogeneous vector, `up(x, y)`;
- `&` spans objects through points, and `|` intersects objects;
- `o & up(p)` vanishes exactly when the point $p$ lies on the object $o$.

The figures are drawn the way a fragment shader would draw them: every pixel
evaluates `o & up(p)` for its own point $p$. Here a whole pixel grid is a NumPy
array of lattice values, so `o & up(X, Y)` broadcasts one object against all
pixels through the registered NumPy loops.

In [ ]:
import base64, struct, zlib

import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from plotly.offline import init_notebook_mode
from algpy.lattices import exterior3, exterior4, ldual, rdual
from _plotting import slider_controls

init_notebook_mode(connected=False)
pio.templates.default = "plotly_white"

## Drawing an object from its incidence field

For each pixel point $p$, `field(o, p)` is the sum of squared coefficients of
`o & up(p)`: the Slang examples' `norm2`. It is zero exactly on the object and
grows quadratically away from it, so $2f/|\nabla f|$ estimates the distance to
the object to first order. The gradient uses central differences. Each object
is composited onto the picture as an antialiased band of the given width, as
with `smoothstep` and `lerp` in a shader.

The picture spans $y\in[-1,1]$ with the shader's pixel coordinates
`p = (2*id - size)/size.y`. Frames are embedded as PNG images.

In [ ]:
WIDTH, HEIGHT = 480, 360
PIXEL = 2/HEIGHT
xs = (2*np.arange(WIDTH) + 1 - WIDTH)/HEIGHT
ys = (2*np.arange(HEIGHT) + 1 - HEIGHT)/HEIGHT
X, Y = np.meshgrid(xs, ys)  # row 0 is the bottom of the picture


def field(obj, up, x, y):
    values = obj & up(x, y)
    return (type(obj).to_float(values)**2).sum(axis=-1)


def distance(obj, up, step=1e-4):
    f = field(obj, up, X, Y)
    gx = (field(obj, up, X + step, Y) - field(obj, up, X - step, Y))/(2*step)
    gy = (field(obj, up, X, Y + step) - field(obj, up, X, Y - step))/(2*step)
    return 2*f/np.maximum(np.hypot(gx, gy), 1e-300)


def smoothstep(edge0, edge1, x):
    t = np.clip((x - edge0)/(edge1 - edge0), 0, 1)
    return t*t*(3 - 2*t)


def plot(image, obj, up, width, color):
    # Blend color into image where the object is within width of a pixel.
    s = 1 - smoothstep(width - PIXEL, width + PIXEL, np.abs(distance(obj, up)))
    return image + (np.asarray(color, float) - image)*s[..., None]


def png_image(image):
    # Encode an RGB image (row 0 at the bottom) as a Plotly image trace.
    data = (np.clip(image, 0, 1)*255 + .5).astype(np.uint8)
    raw = b"".join(b"\0" + row.tobytes() for row in data)
    def chunk(tag, body):
        return (struct.pack(">I", len(body)) + tag + body
                + struct.pack(">I", zlib.crc32(tag + body)))
    header = struct.pack(">IIBBBBB", data.shape[1], data.shape[0], 8, 2, 0, 0, 0)
    blob = (b"\x89PNG\r\n\x1a\n" + chunk(b"IHDR", header)
            + chunk(b"IDAT", zlib.compress(raw, 9)) + chunk(b"IEND", b""))
    source = "data:image/png;base64," + base64.b64encode(blob).decode()
    return go.Image(source=source, x0=xs[0], dx=PIXEL, y0=ys[0], dy=PIXEL,
                    zsmooth="fast", hoverinfo="skip")


def picture_figure(images, labels, title):
    fig = go.Figure(data=[png_image(images[0])])
    if len(images) > 1:
        fig.frames = [go.Frame(name=str(index), data=[png_image(image)])
                      for index, image in enumerate(images)]
        slider_controls(fig, labels)
    fig.update_xaxes(range=[xs[0] - PIXEL/2, xs[-1] + PIXEL/2], showgrid=False, zeroline=False)
    fig.update_yaxes(range=[ys[0] - PIXEL/2, ys[-1] + PIXEL/2], showgrid=False, zeroline=False,
                     scaleanchor="x", scaleratio=1)
    fig.update_layout(title=title, height=560)
    return fig

## Points and lines in the projective plane

`exterior3` has generators $w, e_x, e_y$ at coefficient positions 1, 2, 4. A
point $(x, y)$ is the right dual of its homogeneous vector $w + x\,e_x + y\,e_y$,
a grade-two element. The meet of two points is the grade-one line
$c\,w + a\,e_x + b\,e_y$ through them, with equation $c + a x + b y = 0$.
The meet of a line and a point is a scalar that vanishes exactly when the
point is on the line, and the join of two lines is their intersection point.
`ldual` undoes `rdual`, recovering homogeneous coordinates.

In [ ]:
def up3(x, y):
    # Points (x, y), scalars or arrays: rdual(w + x*e_x + y*e_y).
    x, y = np.broadcast_arrays(np.asarray(x, float), np.asarray(y, float))
    coefficients = np.zeros(x.shape + (exterior3.N,))
    coefficients[..., 1], coefficients[..., 2], coefficients[..., 4] = 1, x, y
    return rdual(exterior3.from_float(coefficients))


def coordinates3(point):
    homogeneous = exterior3.to_float(ldual(point))
    return homogeneous[[2, 4]]/homogeneous[1]


a, b = up3(1., .5), up3(0., -.5)
ab = a & b
assert ab & a == 0 and ab & b == 0
assert ab.grade[1] == ab
print("Line through a and b:", ab)

c, d = up3(-.9, .7), up3(.5, -.75)
cd = c & d
crossing = ab | cd
np.testing.assert_allclose(coordinates3(crossing), [5/38, -7/19], atol=1e-14)
assert abs((ab & crossing)[0]) < 1e-15 and abs((cd & crossing)[0]) < 1e-15
print("Intersection of ab and cd:", coordinates3(crossing))

In [ ]:
angles3 = np.linspace(0, np.pi, 25)[:-1]


def projective_picture(angle):
    # The line through c turns about c; its intersection with ab follows.
    turning = c & up3(-.9 + np.cos(angle), .7 + np.sin(angle))
    crossing = ab | turning
    image = np.ones((HEIGHT, WIDTH, 3))
    image = plot(image, ab, up3, .01, (0, 0, 1))
    image = plot(image, turning, up3, .01, (.55, .55, .55))
    for point in (a, b, c):
        image = plot(image, point, up3, .02, (0, .65, 0))
    if abs(exterior3.to_float(ldual(crossing))[1]) > 1e-9:  # skip parallel lines
        assert abs((turning & crossing)[0]) < 1e-12
        image = plot(image, crossing, up3, .025, (.86, .15, .47))
    return image


fig = picture_figure([projective_picture(angle) for angle in angles3],
                     [f"{np.degrees(angle):.0f}°" for angle in angles3],
                     "exterior3: the line ab, a line turning about c, and their intersection ab | turning")
fig.show(auto_play=False)

## Circles through three points

Lift a point to the paraboloid instead: `exterior4` has generators
$w, e_x, e_y, e_r$, and $(x, y)$ becomes the right dual of
$w + x\,e_x + y\,e_y + (x^2 + y^2)\,e_r$, a grade-three element. A grade-one
element $A\,w + B\,e_x + C\,e_y + D\,e_r$ is the circle
$A + B x + C y + D(x^2 + y^2) = 0$; lines are the circles with $D = 0$.

So the meet of three points `a & b & c` is the circle through them, with
center $-(B, C)/(2D)$ and squared radius $|\text{center}|^2 - A/D$. The join
`line | circle` of a line and a circle is a grade-two element whose
incidence field vanishes at their two intersection points.

In [ ]:
def up4(x, y):
    # Points (x, y), scalars or arrays: rdual(w + x*e_x + y*e_y + (x^2+y^2)*e_r).
    x, y = np.broadcast_arrays(np.asarray(x, float), np.asarray(y, float))
    coefficients = np.zeros(x.shape + (exterior4.N,))
    coefficients[..., 1], coefficients[..., 2] = 1, x
    coefficients[..., 4], coefficients[..., 8] = y, x*x + y*y
    return rdual(exterior4.from_float(coefficients))


def circle_vector(A, B, C, D):
    # The grade-one element A*w + B*e_x + C*e_y + D*e_r.
    coefficients = np.zeros(exterior4.N)
    coefficients[[1, 2, 4, 8]] = A, B, C, D
    return exterior4(*coefficients)


def circle_data(circle):
    A, B, C, D = exterior4.to_float(circle)[[1, 2, 4, 8]]
    center = -np.array([B, C])/(2*D)
    return center, np.sqrt(center @ center - A/D)


def scene4(t):
    points = [(-.6, 0.), (0., .5), (.6, np.sin(t))]
    a, b, c = (up4(*point) for point in points)
    circle = a & b & c
    line = circle_vector(np.sin(t), 0, 1, 0)  # y = -sin(t)
    return points, (a, b, c), circle, line, line | circle


points, (a4, b4, c4), circle, line, pair = scene4(.7)
assert circle.grade[1] == circle and pair.grade[2] == pair
for point in (a4, b4, c4):
    assert abs((circle & point)[0]) < 1e-14
center, radius = circle_data(circle)
np.testing.assert_allclose([np.hypot(*(np.subtract(p, center))) for p in points], radius, rtol=1e-12)

# The two intersections of the circle with the line y = -sin(t).
height = -np.sin(.7)
intersections = [(center[0] + sign*np.sqrt(radius**2 - (height - center[1])**2), height)
                 for sign in (-1, 1)]
for point in intersections:
    assert np.abs(exterior4.to_float(pair & up4(*point))).max() < 1e-12
    assert np.abs(exterior4.to_float(line & up4(*point))).max() < 1e-12
print("Circle center:", center, "radius:", radius)
print("Line-circle intersections:", np.array(intersections))

In [ ]:
times4 = np.linspace(0, 2*np.pi, 25)[:-1]


def circle_picture(t):
    _, points, circle, line, pair = scene4(t)
    image = np.ones((HEIGHT, WIDTH, 3))
    image = plot(image, circle, up4, .01, (1, 0, 0))
    image = plot(image, line, up4, .01, (0, 0, 1))
    image = plot(image, pair, up4, .03, (.95, .75, 0))
    for point in points:
        image = plot(image, point, up4, .02, (0, .65, 0))
    return image


fig = picture_figure([circle_picture(t) for t in times4], [f"{t:.2f}" for t in times4],
                     "exterior4: the circle a & b & c, the line y = -sin t, and line | circle")
fig.show(auto_play=False)

Every construction above is linear algebra on coefficients: no metric, no
square roots, and no special cases for lines versus circles. Objects that do
not exist in the real plane still have exact algebraic representatives. When
the line misses the circle, `line | circle` is still a valid grade-two
element; its incidence field just has no real zeros, so nothing is drawn.

The fixed geometric algebras offer the same operations alongside their
metric: `pga2` shares `exterior3`'s lattice API, and the conformal model
`cga2` contains the circle constructions of `exterior4` in another basis. The
[projective](02_projective_geometry.ipynb) and
[conformal](03_conformal_geometry.ipynb) notebooks use them together with
rigid motions.

**Try changing** the points, the turning line, or the line through the
circle. Two circles intersect the same way: `circle1 | circle2`.